This notebook is to show that the collected data cannot be used to empirically validate Theorem 2 in the paper, because the group of doctors' agreements with AI is a mixture of doctors' decisions to accept and reject AI suggestions, and the study design could not differentiate between the two scenarios. This is due to the fact that if doctors rejected AI suggestions and made a decision on their own, the final decisions may agree or disagree with AI suggestions since this task was a binary classification problem. 

- doctor_result_data.csv data is the study result dataset

Variables:

1. Independent variable: plausibility, AI correctness
2. Dependent variable: performance metric using accuracy

In [2]:
import pandas as pd
import seaborn as sns
import pingouin as pg
import matplotlib.pyplot as plt
dataset = pd.read_csv('../data/doctor_result_data.csv')

In [3]:
# Get plausibility and AIpred column from raw data
plau_corr = dataset.copy()
plau_corr = plau_corr.rename(columns={'xai_qual': 'Plausibility'})
# get accuracy for AI, dr+AI, dr+XAI
plau_corr['drXAI_correct'] = plau_corr['drXAI'] == plau_corr['gt']
plau_corr['drAI_correct'] = plau_corr['drAI'] == plau_corr['gt']
plau_corr['AI_correct'] = plau_corr['AIpred']==plau_corr['gt']
plau_corr['dr_correct'] = plau_corr['drAlone']==plau_corr['gt']

AI_correct = plau_corr[plau_corr['AI_correct']==True]
AI_incorrect = plau_corr[plau_corr['AI_correct']==False]
drXAI_correct = plau_corr[plau_corr['drXAI_correct']==True]
drAI_correct = plau_corr[plau_corr['drAI_correct']==True]
dr_correct = plau_corr[plau_corr['dr_correct']==True]

# accuracy
dr_acc = dr_correct.shape[0]/plau_corr.shape[0]
AI_acc = AI_correct.shape[0]/plau_corr.shape[0]
drAI_acc = drAI_correct.shape[0]/plau_corr.shape[0]
drXAI_acc = drXAI_correct.shape[0]/plau_corr.shape[0]

# validate theorem 2: m>h
h = dr_acc
m = AI_acc
weight = h*(1.0-m)/(m*(1.0-h)) 
offset = (m-h)/(m*(1.0-h))

In [4]:
AI_acc, dr_acc, drAI_acc, drXAI_acc, weight, offset

(0.8827404479578392,
 0.8247694334650857,
 0.8616600790513834,
 0.8682476943346509,
 0.6252272472225343,
 0.3747727527774657)

Using instance-level accuracy, we conclude that complementary performance was not achieved, because AI_acc = 0.88 < drXAI_acc = 0.86.

This aligns with the study result in the original study paper [1] that conducted statistical test using doctor level accuracy. 

[1] Jin, W., Fatehi, M., Guo, R., & Hamarneh, G. (2024). Evaluating the clinical utility of artificial intelligence assistance and its explanation on the glioma grading task. Artificial intelligence in medicine, 148, 102751. https://doi.org/10.1016/j.artmed.2023.102751


In [5]:
pseudo_E_fr =  AI_correct['Agreement_xai'].value_counts()['Agree']/AI_correct.shape[0]
pseudo_E_fw =  AI_incorrect['Agreement_xai'].value_counts()['Agree']/AI_incorrect.shape[0]
pseudo_E_fw, pseudo_E_fr

(0.5168539325842697, 0.9194029850746268)

In [8]:
# theorem 2 when m>h
pseudo_E_fr > weight * pseudo_E_fw + offset

True

However, the condition of theorem 2 fulfilled. This is because ``Agreement`` in the ```AI_correct``` and ``AI_incorrect`` groups are a mixture of doctors' ``Acceptance`` and ``Rejection`` of AI suggestions. Then ``pseudo_E_fr`` is an overestimate of ``E_fr``, and ``pseudo_E_fw`` is an overestimate of ``E_fw``. Since the original study design did not directly ask participants acceptance and rejection of an AI suggestion, ``E_fr`` and ``E_fw`` data were missing, therefore, using this study data cannot empirically validate theorem 2.

In [10]:
AI_correct['Agreement_xai'].value_counts()

Agree       616
Disagree     54
Name: Agreement_xai, dtype: int64

In [9]:
AI_incorrect['Agreement_xai'].value_counts()

Agree       46
Disagree    43
Name: Agreement_xai, dtype: int64